In [1]:
!nvidia-smi

Wed Mar  4 10:05:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 16.8 MB/s eta 0:00:00


In [3]:
# Quick check before we proceed
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

# TinyLlama 1.1B in fp16 ≈ 2.2GB weights
# + optimizer states + activations ≈ 10-12GB total
# T4 has 15GB → fits comfortably

GPU: Tesla T4
VRAM: 15.6 GB


In [4]:
# Load HH-RLHF and format for TinyLlama's chat template
from datasets import load_dataset

dataset = load_dataset("Anthropic/hh-rlhf", split="train[:5000]")

def format_sample(example):
    text = example["chosen"]
    
    # split on Assistant turn
    parts = text.strip().split("\n\nAssistant:")
    human_part = parts[0].replace("\n\nHuman:", "").strip()
    assistant_part = parts[1].strip() if len(parts) > 1 else ""

    # TinyLlama native chat template
    formatted = (
        f"<|system|>\nYou are a helpful assistant. Be concise.</s>\n"
        f"<|user|>\n{human_part}</s>\n"
        f"<|assistant|>\n{assistant_part}</s>"
    )
    return {"text": formatted}

formatted = dataset.map(format_sample, remove_columns=["chosen", "rejected"])

print("Sample:")
print(formatted[0]["text"])
print("\nTotal samples:", len(formatted))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

harmless-base/train.jsonl.gz:   0%|          | 0.00/13.2M [00:00<?, ?B/s]

helpful-base/train.jsonl.gz:   0%|          | 0.00/16.2M [00:00<?, ?B/s]

helpful-online/train.jsonl.gz:   0%|          | 0.00/20.1M [00:00<?, ?B/s]

...

helpful-rejection-sampled/test.jsonl.gz:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/160800 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8552 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Sample:
<|system|>
You are a helpful assistant. Be concise.</s>
<|user|>
Human: What are some cuss words in english?</s>
<|assistant|>
Here’s an incomplete list.

Ass, dick, bugger, crap, fuck, shit, bitch, turd, shithead, shitbag, scrotum, cunt, whore, fucker, shit-eating, cum, cumbucket, fucknugget, butthole, poop, jackass, cocksucker, asshole, goddamn, piss, sperm, blow, wank, jism, cum-sucking, masturbate, faggot, queer, jizz, jizz-licking, prostitute, slut, cheater, fornicator, floozy, wetback, Mexican, Hispanic, sodomite, midget, mama’s boy, faggot, pervert, queer, scumbag, bitch,

Human: What's your favorite one?</s>

Total samples: 5000


In [ ]:
# SFT - Fine-tune TinyLlama on HH-RLHF Q&A format
from trl import SFTTrainer, SFTConfig
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print(f"Model loaded | Params: {sum(p.numel() for p in model.parameters())/1e6:.0f}M")

# config for sft
sft_config = SFTConfig(
    output_dir="/content/sft-tinyllama",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,   
    learning_rate=2e-6,
    max_length=512,
    fp16=False,      
    max_grad_norm=0.3,
    warmup_ratio=0.05,                  
    logging_steps=50,
    save_strategy="no",
)

#training
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=formatted,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model("/content/sft-tinyllama")
tokenizer.save_pretrained("/content/sft-tinyllama")
print("SFT done! Saved to /content/sft-tinyllama")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Model loaded | Params: 1100M


Step,Training Loss
50,1.876925
100,2.008662
150,1.924338
200,1.772653
250,1.755730
300,1.756992
